In [1]:
from google.cloud import bigquery, storage
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType
from pyspark.sql import functions as F
from datetime import datetime, timedelta, timezone
import json

In [2]:
# Thông tin project
project_id = 'fpt-fresher-495407'

# Khởi tạo kết nối tới Cloud Storage và BigQuery
storage_client = storage.Client()
bq_client = bigquery.Client()

# Cấu hình các file và thư mục configs, temp và landing để làm việc với Cloud Storage
bucket_name = 'healthcare_bucket_longnn'
temp_folder = 'temp/'
landing_path_a = f"gs://{bucket_name}/landing/hospital_a/"
landing_path_b = f"gs://{bucket_name}/landing/hospital_b/"
archive_path_a = f"gs://{bucket_name}/landing/hospital_a/archive/"
archive_path_b = f"gs://{bucket_name}/landing/hospital_b/archive/"
config_file_path = f"gs://{bucket_name}/configs/load_config.csv"


# Cấu hình đường dẫn trong BigQuery
bq_temp_dataset = 'temp_dataset'
bq_audit_table = f"{project_id}.{bq_temp_dataset}.audit_log"
bq_log_table =  f"{project_id}.{bq_temp_dataset}.pipeline_log"
bq_temp_path = f"{bucket_name}/temp/"

# Cấu hình PostgreSQL trong Cloud SQL
## hospital-a instance
hospital_a_db_config = {
    'url': 'jdbc:postgresql://10.76.192.3:5432/hospital_a_db',
    'driver': 'org.postgresql.Driver',
    'user': 'longnn28',
    'password': 'Pythongold@64'
}
## hospital-b instance
hospital_b_db_config = {
    'url': 'jdbc:postgresql://10.76.192.5:5432/hospital_b',
    'driver': 'org.postgresql.Driver',
    'user': 'longnn28',
    'password': 'Pythongold@64'
}

# Khởi tạo SparkSession
spark = SparkSession.builder \
    .appName("HospitalDataIngestion") \
    .getOrCreate()

26/05/07 02:55:00 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
log_entries = []
# Function logging
def log_event(event_type, message, datasource = None, table = None):
    local_tz = timezone(timedelta(hours=7))
    log_entry = {
        'timestamp': datetime.now(timezone.utc).astimezone(local_tz).isoformat(),
        'event_type': event_type,
        'message': message,
        'datasource': datasource,
        'table': table
    }
    log_entries.append(log_entry)
    print(f"{log_entry['timestamp']} - {log_entry['event_type']}: {log_entry['message']} (Datasource: {log_entry['datasource']}, Table: {log_entry['table']})")


In [4]:
# Hàm đọc file config từ Cloud Storage
def read_config_file():
    df = spark.read.format("csv").option("header", "true").load(config_file_path)
    log_event("INFO", f"Config file read successfully")
    return df


In [5]:
# Save logs vào Cloud Storage
def save_logs_to_storage():
    local_tz = timezone(timedelta(hours=7))
    log_file_name = f"pipeline_log_{datetime.now(timezone.utc).astimezone(local_tz).strftime('%Y%m%d_%H%M%S')}.json"
    log_file_path = f"temp/pipeline_logs/{log_file_name}"

    json_data = json.dumps(log_entries, indent=4)

    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(log_file_path)

    blob.upload_from_string(json_data, content_type='application/json')

    print(f"Logs saved to {log_file_path} in Cloud Storage")



In [6]:
# Save logs vào BigQuery
def save_logs_to_bigquery():
    if log_entries:
        log_df = spark.createDataFrame(log_entries)
        log_df.write.format('bigquery') \
            .option('table', bq_log_table) \
            .option('temporaryGcsBucket', bq_temp_path) \
            .mode('append') \
            .save()
        
        print(f"Logs saved to BigQuery table {bq_log_table}")
    else:
        print("No logs to save to BigQuery")


In [7]:
# Move các file đã xử lý vào thư mục archive
def move_existing_files_to_archive(datasource, table):
    blob = list(storage_client.bucket(bucket_name).list_blobs(prefix=f"landing/{datasource}/{table}/"))
    existing_files = [b.name for b in blob if b.name.endswith('.json')]

    if not existing_files:
        log_event("INFO", f"No existing files to move for {datasource}.{table}", datasource, table)
        return
    
    local_tz = timezone(timedelta(hours=7))
    run_ts = datetime.utcnow().astimezone(local_tz).strftime('%Y%m%d_%H%M%S')
    for file in existing_files:
        source_blob = storage_client.bucket(bucket_name).blob(file)

        # Trích xuất ngày từ tên file
        date_part = file.split('_')[-1].split('.')[0]
        year, month, day = date_part[-4:], date_part[2:4], date_part[:2]
        file_name = file.split('/')[-1]

        # Move tới archive
        archive_path = f"landing/{datasource}/archive/{table}/{year}/{month}/{day}/{run_ts}/{file_name}"
        destination_blob = storage_client.bucket(bucket_name).blob(archive_path)

        # Copy và xóa file gốc
        storage_client.bucket(bucket_name).copy_blob(source_blob, storage_client.bucket(bucket_name), archive_path)
        source_blob.delete()

        log_event("INFO", f"Moved file {file} to archive at {archive_path}", datasource, table)



In [8]:
# Lấy watermark từ BigQuery Audit Table
def get_latest_watermark(datasource, table):
    query = f"""
        SELECT MAX(load_timestamp) AS load_timestamp
        FROM `{bq_audit_table}`
        WHERE data_source = '{datasource}' AND tablename = '{table}'
    """
    query_job = bq_client.query(query)
    result = query_job.result()
    for row in result:
        if row.load_timestamp:
            return row.load_timestamp
    return '1900-01-01 00:00:00'
    

In [ ]:
audit_schema = StructType([
    StructField("data_source", StringType(), True),
    StructField("tablename", StringType(), True),
    StructField("load_type", StringType(), True),
    StructField("record_count", LongType(), True),
    StructField("load_timestamp", TimestampType(), True),
    StructField("status", StringType(), True),
])

def extract_and_save_to_landing(datasource, table, load_type, watermark_col):
    try:
        # Chuẩn hóa load_type
        load_type_norm = str(load_type).strip().lower()

        # Chuẩn hóa watermark_col
        watermark_col = None if watermark_col is None else str(watermark_col).strip()

        # Chỉ lấy watermark nếu là incremental
        if load_type_norm == "incremental":
            last_watermark = get_latest_watermark(datasource, table)
        else:
            last_watermark = None

        log_event(
            "INFO",
            f"Starting data extraction for {datasource}.{table} with load type {load_type_norm} and last watermark {last_watermark}",
            datasource,
            table
        )

        # Build query
        if load_type_norm == "full":
            query = f"(SELECT * FROM public.{table}) AS src"

        elif load_type_norm == "incremental":
            if not watermark_col:
                raise ValueError(f"Missing watermark column for incremental table {datasource}.{table}")

            query = f"""
                (
                    SELECT *
                    FROM public.{table}
                    WHERE {watermark_col} > TIMESTAMP '{last_watermark}'
                ) AS src
            """

        else:
            raise ValueError(f"Invalid load_type: {load_type}")

        df = spark.read.format("jdbc") \
            .option("url", hospital_a_db_config['url'] if datasource == 'hospital_a' else hospital_b_db_config['url']) \
            .option("driver", hospital_a_db_config['driver'] if datasource == 'hospital_a' else hospital_b_db_config['driver']) \
            .option("user", hospital_a_db_config['user'] if datasource == 'hospital_a' else hospital_b_db_config['user']) \
            .option("password", hospital_a_db_config['password'] if datasource == 'hospital_a' else hospital_b_db_config['password']) \
            .option("dbtable", query) \
            .load()

        log_event("INFO", f"Data extraction completed for {datasource}.{table}", datasource, table)

        local_tz = timezone(timedelta(hours=7))
        today = datetime.now(timezone.utc).astimezone(local_tz).strftime('%d%m%Y')
        json_file_path = f"landing/{datasource}/{table}/{table}_{today}.json"

        # Tính record count trước
        record_count = df.count()


        bucket = storage_client.bucket(bucket_name)
        blob = bucket.blob(json_file_path)
        blob.upload_from_string(
            df.toPandas().to_json(orient='records', lines=True),
            content_type='application/json'
        )

        log_event("SUCCESS", f"Data saved to Cloud Storage at {json_file_path}", datasource, table)

        audit_timestamp = datetime.now(timezone.utc).astimezone(local_tz)

        # Thêm entry vào audit log
        audit_df = spark.createDataFrame(
        [(
        datasource,
        table,
        load_type_norm,
        int(record_count),
        audit_timestamp,
        "SUCCESS"
        )],
        schema=audit_schema
        )

        audit_df.write.format("bigquery") \
            .option("table", bq_audit_table) \
            .option("temporaryGcsBucket", bq_temp_path) \
            .mode("append") \
            .save()

        log_event("INFO", f"Audit log updated for {datasource}.{table}", datasource, table)

    except Exception as e:
        log_event("ERROR", f"Error processing {datasource}.{table}: {str(e)}", datasource, table)

In [10]:
# Xử lý dữ liệu cho từng bảng theo config
config_df = read_config_file()
for row in config_df.collect():
    if row['is_active'] == '1':
        db = row['database']
        src = row['datasource']
        table = row['tablename']
        load_type = row['loadtype']
        watermark = row['watermark']
        move_existing_files_to_archive(src, table)
        extract_and_save_to_landing(src, table, load_type, watermark)

save_logs_to_storage()
save_logs_to_bigquery()


2026-05-07T09:55:08.357683+07:00 - INFO: Config file read successfully (Datasource: None, Table: None)


2026-05-07T09:55:10.935310+07:00 - INFO: No existing files to move for hospital_a.encounters (Datasource: hospital_a, Table: encounters)
2026-05-07T09:55:11.774028+07:00 - INFO: Starting data extraction for hospital_a.encounters with load type incremental and last watermark 1900-01-01 00:00:00 (Datasource: hospital_a, Table: encounters)
2026-05-07T09:55:12.085973+07:00 - INFO: Data extraction completed for hospital_a.encounters (Datasource: hospital_a, Table: encounters)


2026-05-07T09:55:14.935957+07:00 - SUCCESS: Data saved to Cloud Storage at landing/hospital_a/encounters/encounters_07052026.json (Datasource: hospital_a, Table: encounters)


2026-05-07T09:55:25.528854+07:00 - INFO: Audit log updated for hospital_a.encounters (Datasource: hospital_a, Table: encounters)
2026-05-07T09:55:25.558792+07:00 - INFO: No existing files to move for hospital_a.patients (Datasource: hospital_a, Table: patients)
2026-05-07T09:55:26.321618+07:00 - INFO: Starting data extraction for hospital_a.patients with load type incremental and last watermark 1900-01-01 00:00:00 (Datasource: hospital_a, Table: patients)
2026-05-07T09:55:26.379863+07:00 - INFO: Data extraction completed for hospital_a.patients (Datasource: hospital_a, Table: patients)


2026-05-07T09:55:27.709187+07:00 - SUCCESS: Data saved to Cloud Storage at landing/hospital_a/patients/patients_07052026.json (Datasource: hospital_a, Table: patients)


...

2026-05-07T09:56:17.829028+07:00 - SUCCESS: Data saved to Cloud Storage at landing/hospital_b/providers/providers_07052026.json (Datasource: hospital_b, Table: providers)


2026-05-07T09:56:25.934524+07:00 - INFO: Audit log updated for hospital_b.providers (Datasource: hospital_b, Table: providers)
2026-05-07T09:56:25.966047+07:00 - INFO: No existing files to move for hospital_b.departments (Datasource: hospital_b, Table: departments)
2026-05-07T09:56:25.966090+07:00 - INFO: Starting data extraction for hospital_b.departments with load type full and last watermark None (Datasource: hospital_b, Table: departments)
2026-05-07T09:56:26.009635+07:00 - INFO: Data extraction completed for hospital_b.departments (Datasource: hospital_b, Table: departments)


2026-05-07T09:56:26.362135+07:00 - SUCCESS: Data saved to Cloud Storage at landing/hospital_b/departments/departments_07052026.json (Datasource: hospital_b, Table: departments)


2026-05-07T09:56:31.779071+07:00 - INFO: Audit log updated for hospital_b.departments (Datasource: hospital_b, Table: departments)
Logs saved to temp/pipeline_logs/pipeline_log_20260507_095631.json in Cloud Storage


Logs saved to BigQuery table fpt-fresher-495407.temp_dataset.pipeline_log
